# 01 — NFCorpus Dataset Audit

CAP 6776 Information Retrieval — Hybrid Biomedical IR project.

Downloads NFCorpus (`BeIR/nfcorpus` + `BeIR/nfcorpus-qrels`) from Hugging Face, runs the project's dataset validation checks, and writes `data/processed/{dataset_stats,corpus_stats,query_stats}.json`.

All logic lives in `src/biomedical_ir/data.py` and `src/biomedical_ir/validation.py` — this notebook only orchestrates, per the project's no-duplicated-logic rule.

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
from biomedical_ir.data import load_nfcorpus
from biomedical_ir.validation import validate_nfcorpus
from biomedical_ir.utils import load_config

cfg = load_config('configs/default.yaml')
ds_cfg = cfg['dataset']

data = load_nfcorpus(
    cache_dir=ds_cfg['cache_dir'],
    raw_dir=ds_cfg['raw_dir'],
    splits=tuple(ds_cfg['splits']),
)

print(f"corpus documents: {len(data.corpus)}")
print(f"queries: {len(data.queries)}")
for split, split_qrels in data.qrels.items():
    if split.startswith('_'):
        continue
    print(f"qrels[{split}]: {len(split_qrels)} queries")

In [ ]:
report = validate_nfcorpus(data)
print('Validation OK:', report.ok)
print('Errors:', report.errors)
print('Warnings:', report.warnings)
report.stats

## Sample documents and queries

A quick manual sanity check on top of the automated validation above.

In [ ]:
import itertools

for doc_id, doc in itertools.islice(data.corpus.items(), 3):
    print(doc_id, '|', doc['title'][:80])
print()
for qid, q in itertools.islice(data.queries.items(), 3):
    print(qid, '|', q[:80])

## Write processed stats artifacts

Equivalent to running `python scripts/audit_dataset.py` from the command line.

In [ ]:
subprocess.run([sys.executable, 'scripts/audit_dataset.py', '--from-raw'], check=True)